# 03 · Evaluación

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tomas-pucv/Camanchaca-Predict/blob/main/notebooks/03_evaluate.ipynb)

**Propósito.** Predicciones por imagen, baselines (triviales y físicos sin deep learning),
métricas con IC bootstrap agrupado y comparación pareada ResNet−DeiT, latencia, sonda de fuente
e informe `reports/RESULTADOS.md`.

**Bloqueo de test.** Todo lo que toca `test`/`ext` exige `--final`. Durante el desarrollo solo
se usa `val`; la evaluación final se ejecuta **una vez**, al congelar todo.

**Tiempo esperado (T4).** ~15–25 min (bootstrap 2000 incluido).
**Salidas.** `runs/main/<modelo>/seed<k>/pred_<set>.csv`; `reports/tables/metrics_<set>.md`,
`paired_<set>.md`, `latency.md`, `probe_source.md`, `ohaze_pairs.md`; figuras en `reports/figures/`.

## Configuración común
Misma celda en todos los notebooks (detalles en `00_setup_colab`). En **Colab**: monta Drive,
fija `CAMANCHACA_DATA=/content/data` y `CAMANCHACA_RUNS=/content/drive/MyDrive/camanchaca/runs`,
clona/actualiza el repo en la referencia `REF`, instala el paquete (`pip install -e ".[demo,data]"`,
**sin** reinstalar torch) y desempaqueta los `.tar` de datos de Drive (salvo `NEEDS_DATA = False`). **Fuera de Colab** (`SMOKE=True`)
usa datos sintéticos y pesos aleatorios generados por `camanchaca smoke`: sirve para verificar que
todo corre, **sus números no son resultados**.

In [ ]:
# Configuración común (idéntica en los notebooks 00-06)
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = not IN_COLAB   # True: datos sintéticos + pesos aleatorios (solo verifica que todo corre)
REF = "main"           # rama/tag/commit del repo; para la entrega final usar "v1.0-final"
REPO_URL = "https://github.com/Tomas-pucv/Camanchaca-Predict.git"
DRIVE_DIR = Path("/content/drive/MyDrive/camanchaca")   # tars de datos, corridas, pesos y credenciales

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAMANCHACA_DATA"] = "/content/data"
    os.environ["CAMANCHACA_RUNS"] = str(DRIVE_DIR / "runs")
    REPO = Path("/content/Camanchaca-Predict")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
    !git -C {REPO} fetch -q --all --tags && git -C {REPO} checkout -q {REF} && (git -C {REPO} pull -q --ff-only 2>/dev/null || true)
    %cd {REPO}
    !pip install -q -e ".[demo,data]"   # NO reinstala torch/torchvision: se usan los de Colab
    if not SMOKE and globals().get("NEEDS_DATA", True):
        for tar in sorted((DRIVE_DIR / "data").glob("*.tar")):   # imágenes preprocesadas (notebook 01)
            !camanchaca unpack --tar {tar}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    %cd {REPO}
    if shutil.which("camanchaca") is None:
        print("AVISO: falta el comando `camanchaca`: ejecutar  pip install -e '.[dev,demo]'")

WORK = Path(os.environ.get("CAMANCHACA_SMOKE_DIR", str(REPO / "local_workspace" / "nb_smoke")))
if SMOKE:
    os.environ.update(CAMANCHACA_DATA=str(WORK / "data"), CAMANCHACA_RUNS=str(WORK / "runs"),
                      CAMANCHACA_REPORTS=str(WORK / "reports"), HF_HUB_OFFLINE="1")
    if not (WORK / "smoke.yaml").exists():   # ~4-6 min en CPU, una sola vez
        !camanchaca smoke --workdir {WORK}

sys.path.insert(0, str(REPO / "src"))   # importable sin reiniciar el kernel tras `pip install -e`
from camanchaca.config import load_yaml
from camanchaca.paths import data_root, reports_dir, runs_root

EXP = str(WORK / "smoke.yaml") if SMOKE else "configs/experiment/main.yaml"
EXP_NAME = load_yaml(EXP)["name"]
print(f"Colab={IN_COLAB} | SMOKE={SMOKE} | EXP={EXP}")
print(f"datos={data_root()} | corridas={runs_root()} | reportes={reports_dir()}")

## 1. Validación (se puede repetir)

In [ ]:
!camanchaca predict --exp {EXP} --sets val
!camanchaca baselines --exp {EXP} --sets val
!camanchaca evaluate --exp {EXP} --sets val

In [ ]:
from IPython.display import Image, Markdown, display
TABLES, FIGS = reports_dir() / "tables", reports_dir() / "figures"
for name in ["metrics_val.md", "paired_val.md"]:
    if (TABLES / name).exists(): display(Markdown((TABLES / name).read_text()))
for p in sorted(FIGS.glob("*_val.png")) + sorted(FIGS.glob("learning_curves.png")):
    display(Image(filename=str(p), width=720))

## 2. Latencia (CPU y, si hay, GPU)
Parámetros, GMACs, ms/imagen e imágenes/s; con métricas disponibles dibuja el Pareto
velocidad-exactitud.

In [ ]:
LAT_ARGS = "--device cpu --batch-sizes 1" if SMOKE else "--device auto --batch-sizes 1 64"
!camanchaca latency --exp {EXP} {LAT_ARGS}
if (TABLES / "latency.md").exists(): display(Markdown((TABLES / "latency.md").read_text()))

## 3. Sonda de fuente: ¿el backbone codifica el dataset?
Regresión logística sobre features congeladas que predice la **fuente**. Una exactitud alta
indica que el modelo podría resolver la banda reconociendo el dataset (confusión fuente↔banda).

In [ ]:
!camanchaca probes --exp {EXP}
if (TABLES / "probe_source.md").exists(): display(Markdown((TABLES / "probe_source.md").read_text()))

## 4. Evaluación FINAL (test + externos): UNA sola vez
Solo después de fijar LR, pasos y cualquier umbral con val. Anotar el commit (notebook 00) y
**no** volver a elegir nada tras ver test. En Colab cambiar `FINAL = True` únicamente para esta ronda.

In [ ]:
FINAL = SMOKE   # en Colab: True UNA vez, al final del proyecto
if FINAL:
    !camanchaca predict --exp {EXP} --sets val test ext --final
    !camanchaca baselines --exp {EXP} --sets val test ext --final
    !camanchaca evaluate --exp {EXP} --sets val test ext --final
    !camanchaca probes --exp {EXP} --final

In [ ]:
for name in ["metrics_test.md", "paired_test.md", "metrics_ext.md", "paired_ext.md",
             "ohaze_pairs.md", "eps_sensitivity_test.md"]:
    if (TABLES / name).exists(): display(Markdown(f"### {name}\n\n" + (TABLES / name).read_text()))
for p in sorted(FIGS.glob("*_test.png")) + sorted(FIGS.glob("*_ext.png")):
    display(Image(filename=str(p), width=720))

## 5. Informe y respaldo
`camanchaca report` reúne las tablas en `reports/RESULTADOS.md` (lo enlaza el README). Copiar
`reports/` a Drive y versionarlo desde el PC (`git add reports/ && git commit -m "Resultados ..."`).

In [ ]:
!camanchaca report
if (reports_dir() / "RESULTADOS.md").exists(): display(Markdown((reports_dir() / "RESULTADOS.md").read_text()))
if IN_COLAB:
    !zip -qr {DRIVE_DIR}/reports_{EXP_NAME}.zip reports